# ET-SSL Evaluation & Export Notebook
Load the best checkpoint from notebook 01, run full evaluation on all datasets,
generate methodology discussion plots, and export production artifacts for the detection service.

## 1. Environment Setup

In [ ]:
import os, sys

REPO_URL   = "https://github.com/muro906/Sentinel.git"   # ← update this
REPO_DIR   = "Sentinel"
HYBRID_DIR = f"{REPO_DIR}/hybrid-detection"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL}")
else:
    os.system(f"git -C {REPO_DIR} pull --ff-only")

if HYBRID_DIR not in sys.path:
    sys.path.insert(0, HYBRID_DIR)

import subprocess
subprocess.run(["pip", "install", "-q", "optuna", "joblib", "tqdm"], check=True)
print("Repo ready.")

## 2. Imports

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import joblib
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                               recall_score, accuracy_score,
                               classification_report, roc_curve, confusion_matrix)

# ── Repo imports ────────────────────────────────────────────────────────────
from config.constants import (
    FEATURE_DIM, FEATURE_NAMES,
)
from model.et_ssl import ETSSLModel

In [ ]:
# ── Kaggle API Setup (if datasets not in Drive) ─────────────────────────────
import os, shutil, subprocess

KAGGLE_KEY_PATH = '/content/drive/MyDrive/kaggle.json'
if os.path.exists(KAGGLE_KEY_PATH):
    KAGGLE_DIR = os.path.expanduser('~/.kaggle')
    os.makedirs(KAGGLE_DIR, exist_ok=True)
    shutil.copy(KAGGLE_KEY_PATH, f'{KAGGLE_DIR}/kaggle.json')
    os.chmod(f'{KAGGLE_DIR}/kaggle.json', 0o600)
    subprocess.run(['pip', 'install', '-q', 'kaggle'], check=True)
    print('Kaggle API ready')
else:
    print('Kaggle key not found - assuming datasets already in Drive')

## 3. Configuration & Load Artifacts

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATASET_DIR = '/content/drive/MyDrive/Sentinel/datasets'
EXPORT_DIR  = '/content/drive/MyDrive/Sentinel/checkpoints'
PROD_DIR    = f'{EXPORT_DIR}/production'
os.makedirs(PROD_DIR, exist_ok=True)

# ── Select best model from all three trained datasets ────────────────────────
# Reads each meta_{ds}.json saved by notebook 01 and picks the one with the
# highest val_auc. This avoids hardcoding a single dataset as "best".
DATASETS = ['darknet', 'ids2018', 'unsw']
best_auc       = -1
DATASET_CHOICE = None
best_meta      = None

for ds in DATASETS:
    meta_path = f'{EXPORT_DIR}/meta_{ds}.json'
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        val_auc = meta.get('val_auc', 0)
        print(f'{ds}: val_auc = {val_auc:.4f}')
        if val_auc > best_auc:
            best_auc       = val_auc
            DATASET_CHOICE = ds
            best_meta      = meta
    else:
        print(f'{ds}: metadata not found')

if DATASET_CHOICE is None:
    raise FileNotFoundError('No trained model metadata found. Run notebook 01 for all datasets first.')

print(f'\nBest model: {DATASET_CHOICE} (val_auc = {best_auc:.4f})')

PLOTS_DIR = f'{EXPORT_DIR}/plots/{DATASET_CHOICE}'
os.makedirs(PLOTS_DIR, exist_ok=True)

# ── Extract fields from best meta (used by Cell 9 and beyond) ────────────────
# These variables were missing — Cell 9 uses all four and would crash without them.
meta        = best_meta
BEST_CONFIG = meta['best_config']
EMBED_DIM   = meta['embed_dim']
PROJ_DIM    = meta.get('proj_dim', 32)
THRESHOLD   = meta['threshold']

print(f'Embed dim : {EMBED_DIM}')
print(f'Threshold : {THRESHOLD:.6f}')
print(f'Val AUC   : {meta["val_auc"]:.4f}')
print(f'Test AUC  : {meta["test_auc"]:.4f}')


## 4. Rebuild Model & Load Weights

In [ ]:
# ETSSLModel now accepts hidden_dims — reconstruct from best config
hidden_dims = tuple(BEST_CONFIG.get('hidden_dims', [128, 256, 128]))

model = ETSSLModel(
    hidden_dims=hidden_dims,
    embed_dim=EMBED_DIM,
    proj_dim=PROJ_DIM,
    dropout=BEST_CONFIG.get('dropout', 0.3),
).to(DEVICE)

state = torch.load(f'{EXPORT_DIR}/encoder_{DATASET_CHOICE}.pt', map_location=DEVICE)
model.load_state_dict(state)
model.eval()

# Load scaler and centroid
scaler   = joblib.load(f'{EXPORT_DIR}/scaler_{DATASET_CHOICE}.joblib')
centroid = np.load(f'{EXPORT_DIR}/centroid_{DATASET_CHOICE}.npy')

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total_params:,} parameters")
print(f"Architecture: {hidden_dims} → {EMBED_DIM} → {PROJ_DIM}")
print(f"Centroid shape: {centroid.shape}")

## 5. Evaluate on All Datasets

In [ ]:
# ── Evaluate on held-out test splits from all three datasets ─────────────────
# Each test split was saved by notebook 01 (X_test_{ds}.npy, y_test_{ds}.npy).
# These rows were set aside before training began and were never seen during
# training, validation, or threshold calibration — they are the correct data
# to use for final evaluation.
# The full merged CSV is NOT used here to avoid data leakage from train/val rows.

@torch.no_grad()
def get_embeddings(model, X_np, device, bs=1024):
    model.eval()
    dl = DataLoader(torch.from_numpy(X_np).float(), batch_size=bs)
    return torch.cat([model.encode(b.to(device)) for b in dl]).cpu().numpy()

def evaluate_test_split(ds_name, model, centroid, threshold, device):
    """
    Load the pre-saved test split for ds_name and score it with the given model.
    Returns None if the test split files are missing.
    """
    X_path = f'{EXPORT_DIR}/X_test_{ds_name}.npy'
    y_path = f'{EXPORT_DIR}/y_test_{ds_name}.npy'

    if not os.path.exists(X_path) or not os.path.exists(y_path):
        print(f"  {ds_name}: test split not found at {X_path} — skipping.")
        return None

    X_test = np.load(X_path).astype(np.float32)
    y_test = np.load(y_path)
    print(f"  {ds_name}: loaded test split — {X_test.shape[0]:,} samples  "
          f"(anomaly rate: {100*y_test.mean():.1f}%)")

    z      = get_embeddings(model, X_test, device)
    scores = ((z - centroid[None, :]) ** 2).sum(axis=1)
    preds  = (scores > threshold).astype(int)

    auc = roc_auc_score(y_test, scores) if y_test.sum() > 0 and y_test.sum() < len(y_test) else float('nan')

    return {
        'y': y_test, 'scores': scores, 'preds': preds,
        'AUC':       round(auc, 4),
        'Accuracy':  round(accuracy_score(y_test, preds), 4),
        'Precision': round(precision_score(y_test, preds, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, preds, zero_division=0), 4),
        'F1':        round(f1_score(y_test, preds, zero_division=0), 4),
        'N':         len(y_test),
        'anomaly_pct': round(100 * y_test.mean(), 2),
    }

# Run evaluation across all three held-out test sets
results = {}
for ds_name in DATASETS:
    print(f"\nEvaluating on {ds_name} test split...")
    result = evaluate_test_split(ds_name, model, centroid, THRESHOLD, DEVICE)
    if result is not None:
        results[ds_name] = result
        print(classification_report(result['y'], result['preds'],
                                    target_names=['Normal', 'Anomaly']))

print("\n" + "=" * 60)
print("CROSS-DATASET TEST EVALUATION SUMMARY")
print("=" * 60)
summary = {k: {m: v[m] for m in ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1', 'N', 'anomaly_pct']}
           for k, v in results.items()}
print(pd.DataFrame(summary).T.to_string())


## 6. Evaluation Plots (Methodology Discussion)

In [ ]:
n_datasets = len(results)
if n_datasets == 0:
    print("No datasets evaluated — skipping plots.")
else:
    fig, axes = plt.subplots(2, n_datasets, figsize=(6 * n_datasets, 10))
    if n_datasets == 1:
        axes = axes.reshape(-1, 1)

    for col, (ds_name, res) in enumerate(results.items()):
        y, scores, preds = res['y'], res['scores'], res['preds']

        # Row 1 — ROC Curve
        ax = axes[0, col]
        fpr, tpr, _ = roc_curve(y, scores)
        ax.plot(fpr, tpr, color='darkorange', linewidth=2,
                label=f"AUC = {res['AUC']:.4f}")
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(f'{ds_name} — ROC Curve', fontweight='bold')
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)

        # Row 2 — Score distribution
        ax2 = axes[1, col]
        ax2.hist(scores[y == 0], bins=80, alpha=0.6, color='steelblue',
                 label='Normal', density=True)
        ax2.hist(scores[y == 1], bins=80, alpha=0.6, color='coral',
                 label='Anomaly', density=True)
        ax2.axvline(THRESHOLD, color='red', linestyle='--', linewidth=2,
                    label=f'Threshold = {THRESHOLD:.4f}')
        ax2.set_xlabel('Anomaly Score ||z - μ||²')
        ax2.set_ylabel('Density')
        ax2.set_title(f'{ds_name} — Score Distribution', fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/cross_dataset_eval.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved to {PLOTS_DIR}/cross_dataset_eval.png")

## 7. Confusion Matrices

In [ ]:
import seaborn as sns

n = len(results)
if n > 0:
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]

    for ax, (ds_name, res) in zip(axes, results.items()):
        cm = confusion_matrix(res['y'], res['preds'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Normal', 'Anomaly'],
                    yticklabels=['Normal', 'Anomaly'])
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        ax.set_title(f'{ds_name} (F1={res["F1"]:.4f})', fontweight='bold')

    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/confusion_matrices.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved to {PLOTS_DIR}/confusion_matrices.png")

## 8. Export Production Artifacts

In [ ]:
# Save full model state (used by detection service to reconstruct ETSSLModel).
# Also save encoder-only weights with prefix stripped for direct ETSSLEncoder loading.
torch.save(model.state_dict(), f'{PROD_DIR}/encoder_weights.pt')
encoder_only_state = {k[len('encoder.'):]: v
                      for k, v in model.state_dict().items()
                      if k.startswith('encoder.')}
torch.save(encoder_only_state, f'{PROD_DIR}/encoder_only_weights.pt')

# Save production meta — matches what detection-service/detector.py expects
prod_meta = {
    'feature_dim':     FEATURE_DIM,
    'embed_dim':       EMBED_DIM,
    'threshold':       THRESHOLD,
    'dropout':         BEST_CONFIG.get('dropout', 0.3),
    'hidden_dims':     list(hidden_dims),
    'dataset_trained': DATASET_CHOICE,
    'val_auc':         meta['val_auc'],
    'test_auc':        meta['test_auc'],
    'eval_results':    {k: {m: v[m] for m in ['AUC','F1','N']}
                        for k, v in results.items() if v is not None},
    'feature_names':   FEATURE_NAMES,
}
with open(f'{PROD_DIR}/model_meta.json', 'w') as f:
    json.dump(prod_meta, f, indent=2)

joblib.dump(scaler, f'{PROD_DIR}/scaler.joblib')
np.save(f'{PROD_DIR}/centroid.npy', centroid)

print("Production artifacts saved to:", PROD_DIR)
print("  encoder_weights.pt")
print("  scaler.joblib")
print("  centroid.npy")
print("  model_meta.json")
print("\nDownload from Drive and place in: detection-service/models/")

## 9. ONNX Export (Optional — Fast CPU Inference)

In [ ]:
try:
    dummy     = torch.randn(1, FEATURE_DIM).to(DEVICE)
    onnx_path = f'{PROD_DIR}/encoder.onnx'
    torch.onnx.export(
        model.encoder, dummy, onnx_path,
        input_names=['features'], output_names=['embedding'],
        dynamic_axes={'features': {0: 'batch'}, 'embedding': {0: 'batch'}},
        opset_version=14,
    )
    print(f"ONNX model saved: {onnx_path}")
    # Verify
    import onnx
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print("ONNX model verification passed.")
except Exception as e:
    print(f"ONNX export failed (non-critical): {e}")